# Synthetic files in the MIMIC-IV layout (testing only)

Create a small synthetic dataset with the MIMIC-IV 3.1 file layout and column names.

Used only to test that extract_mimic.py and run_mimic.py run end to end. The
values are random and carry no clinical meaning; never report results from it.

## Setup
Loads `00_config` (paths) and the library notebooks from this folder.

In [6]:
# Load other notebooks in this folder as modules (all notebooks live in one folder).
import os, sys, types
import nbformat

NB_DIR = os.getcwd()   # open Jupyter in the notebooks folder


def import_notebook(name, filename):
    """Execute the code cells of a notebook in this folder into a module named `name`."""
    if name in sys.modules:
        return sys.modules[name]
    path = os.path.join(NB_DIR, filename)
    mod = types.ModuleType(name)
    mod.__file__ = path
    sys.modules[name] = mod
    for cell in nbformat.read(path, as_version=4).cells:
        if cell.cell_type == "code":
            exec(compile(cell.source, path, "exec"), mod.__dict__)
    return mod


config = import_notebook("config", "00_config.ipynb")

In [7]:
ext = import_notebook("mimic_extract", "06_mimic_extract_cohort.ipynb")
LABS, VENT_ITEMIDS, VITALS = ext.LABS, ext.VENT_ITEMIDS, ext.VITALS

In [8]:
import argparse
import os
import numpy as np
import pandas as pd

### `main` (function)

In [9]:
def main(out, n, seed=0):
    rng = np.random.default_rng(seed)
    os.makedirs(os.path.join(out, "hosp"), exist_ok=True)
    os.makedirs(os.path.join(out, "icu"), exist_ok=True)
    sid = np.arange(10000000, 10000000 + n)
    groups = rng.choice(["2008 - 2010", "2011 - 2013", "2014 - 2016", "2017 - 2019", "2020 - 2022"], n)
    anchor_year = rng.integers(2110, 2190, n)
    pd.DataFrame(dict(subject_id=sid, gender=rng.choice(["M", "F"], n), anchor_age=rng.integers(18, 91, n),
                      anchor_year=anchor_year, anchor_year_group=groups, dod=None)) \
        .to_csv(os.path.join(out, "hosp", "patients.csv.gz"), index=False)
    hadm = sid + 20000000
    stay = sid + 30000000
    intime = pd.to_datetime(anchor_year.astype(str) + "-03-01") + pd.to_timedelta(rng.integers(0, 300, n), "D")
    los = rng.gamma(2, 2, n)
    sev = rng.normal(size=n)
    death = rng.binomial(1, 1 / (1 + np.exp(-(sev * 1.5 - 2))))
    deathtime = np.where(death == 1, (intime + pd.to_timedelta(rng.integers(30, 400, n), "h")).astype(str), None)
    pd.DataFrame(dict(subject_id=sid, hadm_id=hadm, admittime=intime - pd.Timedelta("5h"),
                      dischtime=intime + pd.to_timedelta(los + 2, "D"), deathtime=deathtime,
                      admission_type=rng.choice(["EW EMER.", "URGENT", "ELECTIVE", "OBSERVATION ADMIT"], n),
                      admission_location="EMERGENCY ROOM", insurance="Other", race="WHITE",
                      hospital_expire_flag=death)).to_csv(os.path.join(out, "hosp", "admissions.csv.gz"), index=False)
    pd.DataFrame(dict(subject_id=sid, hadm_id=hadm, stay_id=stay,
                      first_careunit=rng.choice(["Medical Intensive Care Unit (MICU)", "Surgical Intensive Care Unit (SICU)",
                                                 "Cardiac Vascular Intensive Care Unit (CVICU)"], n),
                      last_careunit="MICU", intime=intime, outtime=intime + pd.to_timedelta(los, "D"), los=los)) \
        .to_csv(os.path.join(out, "icu", "icustays.csv.gz"), index=False)
    centre = dict(hr=85, sbp=120, map=80, rr=18, spo2=96, temp_c=37, temp_f=98.6, gcs_eye=3, gcs_verbal=4, gcs_motor=5)
    ce = []
    for name, (ids, _) in VITALS.items():
        for _ in range(4):
            v = centre[name] + rng.normal(0, 0.08 * centre[name], n) + (sev * 3 if name == "hr" else 0)
            if name.startswith("gcs"):
                v = np.clip(np.round(v), 1, 6)
            ce.append(pd.DataFrame(dict(subject_id=sid, hadm_id=hadm, stay_id=stay, caregiver_id=1,
                                        charttime=intime + pd.to_timedelta(rng.uniform(0, 30, n), "h"),
                                        storetime=intime, itemid=rng.choice(ids, n), value=v.round(1),
                                        valuenum=v.round(1), valueuom="", warning=0)))
    pd.concat(ce).to_csv(os.path.join(out, "icu", "chartevents.csv.gz"), index=False)
    lcentre = dict(creatinine=1.1, bun=20, sodium_min=139, sodium_max=139, potassium=4.2, bicarbonate=24,
                   glucose=130, hemoglobin=11, wbc=10, platelets=220, lactate=1.8, bilirubin=0.9, albumin=3.3,
                   inr=1.2, aniongap=14)
    le = []
    for name, (ids, _, _) in LABS.items():
        keep = rng.random(n) < (0.5 if name in ("lactate", "albumin", "bilirubin") else 0.95)
        v = lcentre[name] * np.exp(rng.normal(0, 0.15, n) + (0.1 * sev if name in ("creatinine", "lactate") else 0))
        le.append(pd.DataFrame(dict(labevent_id=0, subject_id=sid, hadm_id=hadm, specimen_id=0,
                                    itemid=rng.choice(ids, n), order_provider_id=None,
                                    charttime=intime + pd.to_timedelta(rng.uniform(-6, 30, n), "h"), storetime=intime,
                                    value=v.round(2), valuenum=v.round(2), valueuom="", ref_range_lower=None,
                                    ref_range_upper=None, flag=None, priority=None, comments=None))[keep])
    pd.concat(le).to_csv(os.path.join(out, "hosp", "labevents.csv.gz"), index=False)
    m = rng.random(n) < 0.35
    pd.DataFrame(dict(subject_id=sid[m], hadm_id=hadm[m], stay_id=stay[m], caregiver_id=1,
                      starttime=(intime + pd.to_timedelta(rng.uniform(0, 30, n), "h"))[m], endtime=intime[m],
                      storetime=intime[m], itemid=VENT_ITEMIDS[0], value=1, valueuom="None")) \
        .to_csv(os.path.join(out, "icu", "procedureevents.csv.gz"), index=False)
    print("mock written to", out)

## Run
Writes mock files to `OUTPUT_DIR/mock_mimic`. To test notebooks 06-07 on them, set the environment variables `MIMIC_ROOT` to that folder and `OUTPUT_DIR` to a scratch folder before starting Jupyter. Never report results from mock data.

In [10]:
MOCK_DIR = os.path.join(config.OUTPUT_DIR, "mock_mimic")
main(MOCK_DIR, 1500)

mock written to D:\New_Research_2027\Clinical_XAI_Faithfulness\outputs\mock_mimic
